In [0]:

catalog = "meu_catalog"
schema = "default"
volume = "inputs"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"

print(volume_path)

In [0]:
display(dbutils.fs.ls(volume_path))

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS meu_catalog.bronze
""")

In [0]:
display(
    spark.sql("SHOW SCHEMAS IN meu_catalog")
)

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
df_movies_info = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/meu_catalog/default/inputs/movies_info_TMDB_IMDB.csv")
)

In [0]:
display(df_movies_info)

In [0]:
df_movies_info_bronze = (
    df_movies_info
    .withColumn("ingestion_datetime", current_timestamp())
)

display(df_movies_info_bronze)

In [0]:
(
    df_movies_info_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_movies_info")
)

display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_movies_info
        LIMIT 10
    """)
)

display(
    spark.sql("""
        SHOW TABLES IN meu_catalog.bronze
    """)
)

In [0]:
df_movies_financials = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/meu_catalog/default/inputs/movies_financials_IMDB_TMDB.csv")
)

df_movies_financials_bronze = (
    df_movies_financials
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_financials_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_movies_financials")
)



In [0]:
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_movies_financials
        LIMIT 10
    """)
)

spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_movies_financials
""").show(truncate=False)

In [0]:
df_movies_metrics = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/meu_catalog/default/inputs/movies_metrics_IMDB_TMDB.csv")
)

df_movies_metrics_bronze = (
    df_movies_metrics
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_metrics_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_movies_metrics")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_movies_metrics
        LIMIT 10
    """)
)

spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_movies_metrics
""").show(truncate=False)

In [0]:
df_credits_and_tags = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/meu_catalog/default/inputs/credits_and_tags_IMDB_TMDB.csv")
)

display(df_credits_and_tags)

df_credits_and_tags_bronze = (
    df_credits_and_tags
    .withColumn("ingestion_datetime", current_timestamp())
)

display(df_credits_and_tags_bronze)

(
    df_credits_and_tags_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_credits_and_tags")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_credits_and_tags
        LIMIT 10
    """)
)

spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_credits_and_tags
""").show(truncate=False)

In [0]:
df_movies_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/meu_catalog/default/inputs/movies_reviews.csv")
)

display(df_movies_reviews)

df_movies_reviews_bronze = (
    df_movies_reviews
    .withColumn("ingestion_datetime", current_timestamp())
)

display(df_movies_reviews_bronze)

(
    df_movies_reviews_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_movies_reviews")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_movies_reviews
        LIMIT 10
    """)
)

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN meu_catalog.bronze
    """)
)

API

In [0]:
from datetime import datetime, timedelta

# Data final: hoje
data_fim_default = datetime.now()

# Data inicial: 7 dias antes
data_inicio_default = data_fim_default - timedelta(days=7)

dbutils.widgets.text(
    "data_inicio",
    data_inicio_default.strftime("%m-%d-%Y"),
    "Data início (MM-DD-AAAA)"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_default.strftime("%m-%d-%Y"),
    "Data fim (MM-DD-AAAA)"
)

In [0]:
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")

In [0]:
import json

with open("/Volumes/meu_catalog/default/inputs/cotacao_dolar_manual.json", "r", encoding="utf-8") as f:
    dados_cotacao = json.load(f)

print(f"Registros retornados: {len(dados_cotacao['value'])}")

from pyspark.sql import functions as F

df_cotacao_raw = spark.createDataFrame(dados_cotacao["value"])
df_cotacao_bronze = df_cotacao_raw.withColumn("ingestion_datetime", F.current_timestamp())

(
    df_cotacao_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_cotacao_dolar")
)

display(df_cotacao_bronze)

''''
import requests

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

print(url)

response = requests.get(url, timeout=30)

print(f"Status HTTP: {response.status_code}")

response.raise_for_status()

dados_cotacao = response.json()

print(f"Registros retornados: {len(dados_cotacao['value'])}")
'''''
#print("LOL")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_cotacao_dolar
        ORDER BY dataHoraCotacao
    """)
)

spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_cotacao_dolar
""").show(truncate=False)

display(
    spark.sql("""
        SHOW TABLES IN meu_catalog.bronze
    """)
)